In [1]:
%env MUJOCO_GL=osmesa
import functools
import json
import math
import os
import pathlib
import sys

import cv2
import numpy as np
import mediapy as media

from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

from openpi_client.websocket_client_policy import WebsocketClientPolicy

sys.path.append('../py_script')

env: MUJOCO_GL=osmesa


[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /home/signal/workspace/mujoco_test/mujoco_playground/.venv/lib/python3.12/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)


In [3]:
def _get_libero_env(task, resolution, seed):
    """Initializes and returns the LIBERO environment, along with the task description."""
    task_description = task.language
    task_bddl_file = pathlib.Path(get_libero_path("bddl_files")) / task.problem_folder / task.bddl_file
    env_args = {"bddl_file_name": task_bddl_file, "camera_heights": resolution, "camera_widths": resolution, "camera_depths": True}
    env = OffScreenRenderEnv(**env_args)
    env.seed(seed)  # IMPORTANT: seed seems to affect object positions even when using fixed initial state
    return env, task_description

class LiberoEnvMaker:
    def __init__(self, suite: str,
                 render_resolution: int = 512, seed: int = 0,
                 repeats: int = 1):
        benchmark_dict = benchmark.get_benchmark_dict()
        self.task_suite = benchmark_dict[suite]()
        self.repeats = repeats
        self.render_resolution = render_resolution
        self.seed = seed

    def get_num_tasks(self):
        return self.task_suite.n_tasks

    def task_instantiations(self, task_id):
        task = self.task_suite.get_task(task_id)
        print(task, task_id)
        initial_states = self.task_suite.get_task_init_states(task_id)
        env, task_description = _get_libero_env(task, self.render_resolution, self.seed)
        for episode_idx in range(self.repeats):
            env.reset()
            obs = env.set_init_state(initial_states[episode_idx])
            yield obs, env, task_description

import mujoco

def add_visual_point(scn, pos, rgba=[1, 0, 0, 1], radius=0.01):
    """Add a sphere marker at a 3D position to an MjvScene.
    
    Args:
    scn: mujoco.MjvScene object (e.g. env.sim._render_context_offscreen.scn)
    pos: [x, y, z] position
    rgba: [r, g, b, a] color
    radius: sphere radius
    """
    if scn.ngeom >= scn.maxgeom:
        print("WARNING: scene buffer full!")
        return  # scene buffer full
    
    mujoco.mjv_initGeom(
        scn.geoms[scn.ngeom],
        type=mujoco.mjtGeom.mjGEOM_SPHERE,
        size=[radius, 0, 0],
        pos=np.array(pos, dtype=np.float64),
        mat=np.eye(3, dtype=np.float64).flatten(),
        rgba=np.array(rgba, dtype=np.float32),
    )
    scn.ngeom += 1

def patch_env_for_render(env):
    """
    Patch the rendering function of a robosuite environment to add the ability to draw 3d points.
    This code was generated by Claude
    """
    # Get the render context
    render_ctx = env.sim._render_context_offscreen
    # Save original render method (unused)
    original_render = env.sim.render

    def render_with_points(*args, visual_points=[], **kwargs):
        env._visual_points = visual_points
        result = original_render(*args, **kwargs)
        env._visual_points = []
        return result
    env.sim.render = render_with_points

    def patched_render(width, height, camera_id=None, segmentation=False):
        """
        Copy of original function, but we needed to break it up.

        visual_points should be a list of dicts: {"pos": [x,y,z], "rgba": [r,g,b,a], "radius": 0.01}
        """
        # Call original up to mjv_updateScene (we need to replicate the logic)
        viewport = mujoco.MjrRect(0, 0, width, height)
        
        if width > render_ctx.con.offWidth or height > render_ctx.con.offHeight:
            new_width = max(width, render_ctx.model.vis.global_.offwidth)
            new_height = max(height, render_ctx.model.vis.global_.offheight)
            render_ctx.update_offscreen_size(new_width, new_height)
        
        if camera_id is not None:
            if camera_id == -1:
                render_ctx.cam.type = mujoco.mjtCamera.mjCAMERA_FREE
            else:
                render_ctx.cam.type = mujoco.mjtCamera.mjCAMERA_FIXED
                render_ctx.cam.fixedcamid = camera_id
        
        mujoco.mjv_updateScene(
            render_ctx.model._model, render_ctx.data._data,
            render_ctx.vopt, render_ctx.pert, render_ctx.cam,
            mujoco.mjtCatBit.mjCAT_ALL, render_ctx.scn
        )
        
        if segmentation:
            render_ctx.scn.flags[mujoco.mjtRndFlag.mjRND_SEGMENT] = 1
            render_ctx.scn.flags[mujoco.mjtRndFlag.mjRND_IDCOLOR] = 1
        
        # --- Add visual points here ---
        for pt in env._visual_points:
            add_visual_point(render_ctx.scn, pt["pos"], pt.get("rgba", [1,0,0,1]), pt.get("radius", 0.01))
        
        mujoco.mjr_render(viewport=viewport, scn=render_ctx.scn, con=render_ctx.con)
        
        if segmentation:
            render_ctx.scn.flags[mujoco.mjtRndFlag.mjRND_SEGMENT] = 0
            render_ctx.scn.flags[mujoco.mjtRndFlag.mjRND_IDCOLOR] = 0

    render_ctx.render = patched_render

def get_cam_pose(env, cam_name="agentview"):
    cam_id = env.sim.model.camera_name2id(cam_name)
    return (env.sim.model.cam_pos[cam_id], env.sim.model.cam_quat[cam_id])
    
def set_cam_pose(env, pose, cam_name="agentview"):
    cam_id = env.sim.model.camera_name2id(cam_name)
    env.sim.model.cam_pos[cam_id] = pose[0]
    env.sim.model.cam_quat[cam_id] = pose[1]

In [ ]:
policy = WebsocketClientPolicy(port=9898)

In [22]:
rollout = []
frames = []
wrist_frames = []
explain_frames = []

libero_envs = LiberoEnvMaker('libero_10')

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [23]:
def run(policy, iters=120, init=True, override_task=None, early_exit=True):
    global obs, env
    # STARTUP section
    if init:
        rollout.clear()
        frames.clear()
        wrist_frames.clear()
        instance = next(libero_envs.task_instantiations(9))
        obs, env, task_description = instance
        if override_task is not None:
            task_description = override_task
        patch_env_for_render(env)
        policy.initialize(obs, task_description)
    
    vla_output = policy.infer(obs)
    actions = vla_output['actions']
    rollout.append(obs)
    
    trajectory_idx = 0
    for i in range(iters):
        act = np.copy(actions[trajectory_idx])
        obs, reward, done, info = env.step(act)
        rollout.append(obs)
        frame = np.copy(obs['agentview_image'][::-1, ::-1, :])
        frames.append(frame)
        wrist_frames.append(obs['robot0_eye_in_hand_image'][::-1, ::-1, :])
        if done and early_exit:
            break
        trajectory_idx += 1
        if trajectory_idx == (len(actions)//2):
            vla_output = policy.infer(obs)
            actions = vla_output['actions']
            trajectory_idx = 0
#run(policy, override_task="pick up the white mug")
run(policy, iters=60)
# policy.initialize(obs, "pick up the white mug")
# run(policy, iters=120, init=False, early_exit=False)
# run(policy, iters=120, init=False)

freq = 20
#save_cam_pose = get_cam_pose(env)
img = obs['agentview_image'][::-1, ::-1, :]
import matplotlib.pyplot as plt
plt.figure(0)
plt.clf()
plt.imshow(img)
plt.figure(1)
plt.clf()
plt.imshow(obs['agentview_depth'][::-1, ::-1, :])
plt.show()
media.write_video(f'franka_90.mp4', frames, fps=freq)
media.write_video(f'franka_90_wrist.mp4', wrist_frames, fps=freq)